In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import Libraries

In [2]:
import os
import re
import gc
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(100)
np.random.seed(100)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(100)

print("Device:", device)

Device: cuda


# Configuration and Settings

In [3]:

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

DEBERTA_MODEL_PATH = "microsoft/deberta-v3-base"
ROBERTA_MODEL_PATH = "roberta-base"

RUN_SCRATCH_TRAINING = True
RUN_TRANSFORMER_TRAINING = True

BIGRU_EPOCHS = 10
BIGRU_BATCH_SIZE = 32
BIGRU_LR = 0.001
BIGRU_EMBED_DIM = 256
BIGRU_HIDDEN_DIM = 256
BIGRU_DROPOUT = 0.30

TRANSFORMER_EPOCHS = 4
TRANSFORMER_BATCH_SIZE = 2
DEBERTA_LR = 5e-6
ROBERTA_LR = 1e-5
GRAD_ACCUM_STEPS = 8

MIN_USABLE_MAP3 = 0.40

LABEL_MAP = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
OPTIONS = np.array(["A", "B", "C", "D", "E"])

## Loading  the dataset

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

train_df_clean = train_df.drop_duplicates(
    subset=["prompt", "A", "B", "C", "D", "E"]
).reset_index(drop=True)

print("Clean train shape:", train_df_clean.shape)
train_df_clean.head()


Train shape: (2000, 8)
Test shape : (500, 7)
Clean train shape: (1817, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Text preprocessing and MAP@3 metric

In [5]:
def preprocess_clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def compute_map3(logits, targets):
    top3 = np.argsort(logits, axis=1)[:, ::-1][:, :3]
    score = 0.0

    for i in range(len(targets)):
        target = targets[i]
        predictions = top3[i]

        for rank, prediction in enumerate(predictions):
            if prediction == target:
                score += 1.0 / (rank + 1)
                break

    return score / len(targets)




# Create five training folds and RAG context

In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=100)
folds_data = []

for fold, (train_idx, val_idx) in enumerate(kf.split(train_df_clean)):
    train_fold = train_df_clean.iloc[train_idx].copy()
    val_fold = train_df_clean.iloc[val_idx].copy()

    fold_vectorizer = TfidfVectorizer(
        max_features=5000,
        stop_words="english"
    )

    fold_prompt_vectors = fold_vectorizer.fit_transform(
        train_fold["prompt"].astype(str)
    )

    def get_similar_queries_fold(query, top_k=3):
        query_vector = fold_vectorizer.transform([str(query)])
        similarities = cosine_similarity(
            query_vector,
            fold_prompt_vectors
        )[0]

        top_indices = np.argsort(similarities)[::-1][:top_k]
        return train_fold.iloc[top_indices]

    def expand_with_context_fold(dataframe):
        augmented_prompts = []

        for _, row in dataframe.iterrows():
            query = str(row["prompt"])
            similar_rows = get_similar_queries_fold(query, top_k=3)

            context = ""

            for _, similar_row in similar_rows.iterrows():
                correct_answer_text = str(
                    similar_row[similar_row["answer"]]
                )

                context += (
                    f"Q: {similar_row['prompt']} "
                    f"A: {correct_answer_text}\n"
                )

            augmented_prompts.append(
                context + "Question: " + query
            )

        new_dataframe = dataframe.copy()
        new_dataframe["prompt"] = augmented_prompts

        return new_dataframe

    print(f"Preparing fold {fold + 1}...")

    folds_data.append({
        "train": train_fold,
        "val": val_fold,
        "train_aug": expand_with_context_fold(train_fold),
        "val_aug": expand_with_context_fold(val_fold)
    })


Preparing fold 1...
Preparing fold 2...
Preparing fold 3...
Preparing fold 4...
Preparing fold 5...


##  Create retrieval context for the test set

In [7]:
test_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

test_prompt_vectors = test_vectorizer.fit_transform(
    train_df_clean["prompt"].astype(str)
)


def get_similar_queries_test(query, top_k=3):
    query_vector = test_vectorizer.transform([str(query)])

    similarities = cosine_similarity(
        query_vector,
        test_prompt_vectors
    )[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    return train_df_clean.iloc[top_indices]


def expand_with_context_test(dataframe):
    augmented_prompts = []

    for _, row in dataframe.iterrows():
        query = str(row["prompt"])
        similar_rows = get_similar_queries_test(query, top_k=3)

        context = ""

        for _, similar_row in similar_rows.iterrows():
            correct_answer_text = str(
                similar_row[similar_row["answer"]]
            )

            context += (
                f"Q: {similar_row['prompt']} "
                f"A: {correct_answer_text}\n"
            )

        augmented_prompts.append(
            context + "Question: " + query
        )

    new_dataframe = dataframe.copy()
    new_dataframe["prompt"] = augmented_prompts

    return new_dataframe


print("Preparing augmented test data...")
test_df_aug = expand_with_context_test(test_df)


Preparing augmented test data...


# Scratch BiGRU model
# Prepare data for the scratch model

In [8]:
class TextVocabulary:
    def __init__(self, texts, max_size=12000):
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}

        words = []

        for text in texts:
            words.extend(
                preprocess_clean(text).split()
            )

        from collections import Counter

        counts = Counter(words)

        for word, _ in counts.most_common(max_size - 2):
            self.word2idx[word] = len(self.word2idx)

    def __len__(self):
        return len(self.word2idx)

    def encode(self, text, max_len=128):
        tokens = preprocess_clean(text).split()

        sequence = [
            self.word2idx.get(token, 1)
            for token in tokens[:max_len]
        ]

        if len(sequence) < max_len:
            sequence += [0] * (max_len - len(sequence))

        return np.array(sequence)


class CustomMCQDataset(Dataset):
    def __init__(
        self,
        dataframe,
        vocabulary,
        max_prompt_len=128,
        max_option_len=64,
        is_test=False
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.vocabulary = vocabulary
        self.max_prompt_len = max_prompt_len
        self.max_option_len = max_option_len
        self.is_test = is_test

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        prompt_sequence = self.vocabulary.encode(
            row["prompt"],
            self.max_prompt_len
        )

        option_sequences = []

        for option_column in ["A", "B", "C", "D", "E"]:
            option_sequences.append(
                self.vocabulary.encode(
                    row[option_column],
                    self.max_option_len
                )
            )

        item = {
            "prompt": torch.tensor(
                prompt_sequence,
                dtype=torch.long
            ),
            "options": torch.tensor(
                np.array(option_sequences),
                dtype=torch.long
            )
        }

        if not self.is_test:
            item["label"] = torch.tensor(
                LABEL_MAP[row["answer"]],
                dtype=torch.long
            )

        return item


##  Build the BiGRU model from scratch


In [9]:
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dimension):
        super().__init__()
        self.attention = nn.Linear(hidden_dimension, 1)

    def forward(self, hidden_states, mask):
        scores = self.attention(hidden_states).squeeze(-1)
        scores = scores.masked_fill(mask == 0,torch.finfo(scores.dtype).min)

        weights = torch.softmax(scores, dim=1).unsqueeze(-1)

        return torch.sum(
            hidden_states * weights,
            dim=1
        )


class BiGRUMCQModel(nn.Module):
    def __init__(
        self,
        vocabulary_size,
        embedding_dimension=256,
        hidden_dimension=256,
        dropout=0.30
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dimension,
            padding_idx=0
        )

        self.prompt_gru = nn.GRU(
            embedding_dimension,
            hidden_dimension,
            batch_first=True,
            bidirectional=True
        )

        self.option_gru = nn.GRU(
            embedding_dimension,
            hidden_dimension,
            batch_first=True,
            bidirectional=True
        )

        self.prompt_attention = AttentionLayer(
            hidden_dimension * 2
        )

        self.option_attention = AttentionLayer(
            hidden_dimension * 2
        )

        combined_dimension = hidden_dimension * 2 * 4

        self.classifier = nn.Sequential(
            nn.Linear(
                combined_dimension,
                hidden_dimension * 2
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dimension * 2, 1)
        )

    def forward(self, prompt_ids, option_ids):
        batch_size = prompt_ids.size(0)

        prompt_mask = (prompt_ids != 0).float()
        prompt_embeddings = self.embedding(prompt_ids)

        prompt_outputs, _ = self.prompt_gru(
            prompt_embeddings
        )

        prompt_vector = self.prompt_attention(
            prompt_outputs,
            prompt_mask
        )

        option_ids = option_ids.reshape(
            batch_size * 5,
            option_ids.size(-1)
        )

        option_mask = (option_ids != 0).float()
        option_embeddings = self.embedding(option_ids)

        option_outputs, _ = self.option_gru(
            option_embeddings
        )

        option_vectors = self.option_attention(
            option_outputs,
            option_mask
        )

        option_vectors = option_vectors.reshape(
            batch_size,
            5,
            -1
        )

        prompt_vector = prompt_vector.unsqueeze(1).repeat(
            1,
            5,
            1
        )

        combined_features = torch.cat(
            [
                prompt_vector,
                option_vectors,
                torch.abs(prompt_vector - option_vectors),
                prompt_vector * option_vectors
            ],
            dim=2
        )

        return self.classifier(
            combined_features
        ).squeeze(-1)


In [10]:
# !pip install -q wandb

# from kaggle_secrets import UserSecretsClient
# import os, wandb

# api_key = UserSecretsClient().get_secret("Wandb_key")
# os.environ["Wandb_key"] = api_key

# wandb.login()

In [11]:
# import random

# import wandb

# run = wandb.init(
#     entity="21f2000522-indian-institute-of-technology-madras",
#     project="BiGRU MOdel Rerun",
#     config={
#         "learning_rate": 0.00001,
#         "architecture": "RNN",
#         "dataset": "Kaggle dataset",
#         "epochs": 10,
#     },
# )

# Train and evaluate the BiGRU

In [12]:
def evaluate_bigru(model, data_loader):
    model.eval()

    probabilities_all = []
    targets_all = []
    total_loss = 0.0

    loss_function = nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in data_loader:
            prompt_ids = batch["prompt"].to(device)
            option_ids = batch["options"].to(device)
            targets = batch["label"].to(device)

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):
                scores = model(
                    prompt_ids,
                    option_ids
                )

                loss = loss_function(
                    scores,
                    targets
                )

            probabilities = torch.softmax(
                scores.float(),
                dim=1
            )

            total_loss += loss.item()

            probabilities_all.append(
                probabilities.cpu().numpy()
            )

            targets_all.append(
                targets.cpu().numpy()
            )

    probabilities_all = np.concatenate(
        probabilities_all
    )

    targets_all = np.concatenate(
        targets_all
    )

    accuracy = np.mean(
        np.argmax(
            probabilities_all,
            axis=1
        ) == targets_all
    )

    map3_score = compute_map3(
        probabilities_all,
        targets_all
    )

    return (
        total_loss / len(data_loader),
        accuracy,
        map3_score
    )


def train_bigru_model(
    train_dataframe,
    validation_dataframe,
    save_path="best_bigru_scratch.pth"
):
    vocabulary_texts = []

    for _, row in train_dataframe.iterrows():
        vocabulary_texts.append(str(row["prompt"]))

        for option_column in ["A", "B", "C", "D", "E"]:
            vocabulary_texts.append(
                str(row[option_column])
            )

    vocabulary = TextVocabulary(
        vocabulary_texts,
        max_size=12000
    )

    train_dataset = CustomMCQDataset(
        train_dataframe,
        vocabulary
    )

    validation_dataset = CustomMCQDataset(
        validation_dataframe,
        vocabulary
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BIGRU_BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BIGRU_BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    model = BiGRUMCQModel(
        vocabulary_size=len(vocabulary),
        embedding_dimension=BIGRU_EMBED_DIM,
        hidden_dimension=BIGRU_HIDDEN_DIM,
        dropout=BIGRU_DROPOUT
    ).to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=BIGRU_LR,
        weight_decay=1e-4
    )

    loss_function = nn.CrossEntropyLoss()

    scaler = torch.cuda.amp.GradScaler(
        enabled=torch.cuda.is_available()
    )

    best_map3 = -1.0
    patience = 0

    for epoch in range(1, BIGRU_EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for batch in train_loader:
            prompt_ids = batch["prompt"].to(device)
            option_ids = batch["options"].to(device)
            targets = batch["label"].to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):
                scores = model(
                    prompt_ids,
                    option_ids
                )

                loss = loss_function(
                    scores,
                    targets
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        (
            validation_loss,
            validation_accuracy,
            validation_map3
        ) = evaluate_bigru(
            model,
            validation_loader
        )

    #     run.log(
    #     {"Traning_loss": running_loss / len(train_loader)})

    #     run.log(
    #     {"Val_Accuracy": validation_accuracy, "Val_loss": validation_loss,"Val_MAP3":validation_map3}
    # )

        print(
            f"Epoch {epoch}: "
            f"train_loss={running_loss / len(train_loader):.4f}, "
            f"val_loss={validation_loss:.4f}, "
            f"accuracy={validation_accuracy:.4f}, "
            f"MAP@3={validation_map3:.4f}"
        )

        if validation_map3 > best_map3:
            best_map3 = validation_map3
            patience = 0

            torch.save(
                model.state_dict(),
                save_path
            )

        else:
            patience += 1

        if patience >= 6:
            print("Early stopping")
            break

    model.load_state_dict(
        torch.load(
            save_path,
            map_location=device
        )
    )

    (
        validation_loss,
        validation_accuracy,
        validation_map3
    ) = evaluate_bigru(
        model,
        validation_loader
    )

    return validation_accuracy, validation_map3


# Run Model 1: BiGRU

In [13]:
if RUN_SCRATCH_TRAINING:
    print("=" * 60)
    print("MODEL 1: BiGRU BUILT FROM SCRATCH")
    print("=" * 60)

    best_bigru_accuracy, best_bigru_map3 = train_bigru_model(
        folds_data[0]["train"],
        folds_data[0]["val"]
    )

    print("BiGRU accuracy:", round(best_bigru_accuracy, 4))
    print("BiGRU MAP@3 :", round(best_bigru_map3, 4))

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

else:
    best_bigru_accuracy = 0.0
    best_bigru_map3 = 0.0


# run.finish()


MODEL 1: BiGRU BUILT FROM SCRATCH
Epoch 1: train_loss=0.8469, val_loss=0.1485, accuracy=0.9423, MAP@3=0.9684
Epoch 2: train_loss=0.0753, val_loss=0.0075, accuracy=1.0000, MAP@3=1.0000
Epoch 3: train_loss=0.0083, val_loss=0.0004, accuracy=1.0000, MAP@3=1.0000
Epoch 4: train_loss=0.0003, val_loss=0.0002, accuracy=1.0000, MAP@3=1.0000
Epoch 5: train_loss=0.0001, val_loss=0.0002, accuracy=1.0000, MAP@3=1.0000
Epoch 6: train_loss=0.0000, val_loss=0.0002, accuracy=1.0000, MAP@3=1.0000
Epoch 7: train_loss=0.0000, val_loss=0.0001, accuracy=1.0000, MAP@3=1.0000
Epoch 8: train_loss=0.0000, val_loss=0.0002, accuracy=1.0000, MAP@3=1.0000
Early stopping
BiGRU accuracy: 1.0
BiGRU MAP@3 : 1.0


# Prepare data for transformer models


In [14]:
class TransformerDataset(Dataset):
    def __init__(
        self,
        dataframe,
        tokenizer,
        max_len=128,
        is_test=False
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        prompt = str(row["prompt"])

        input_ids = []
        attention_masks = []
        token_type_ids = []

        for option_column in ["A", "B", "C", "D", "E"]:
            encoded = self.tokenizer(
                prompt,
                str(row[option_column]),
                truncation=True,
                max_length=self.max_len,
                padding="max_length"
            )

            input_ids.append(
                encoded["input_ids"]
            )

            attention_masks.append(
                encoded["attention_mask"]
            )

            if "token_type_ids" in encoded:
                token_type_ids.append(
                    encoded["token_type_ids"]
                )

        item = {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                attention_masks,
                dtype=torch.long
            )
        }

        if len(token_type_ids) > 0:
            item["token_type_ids"] = torch.tensor(
                token_type_ids,
                dtype=torch.long
            )

        if not self.is_test:
            item["label"] = torch.tensor(
                LABEL_MAP[row["answer"]],
                dtype=torch.long
            )

        return item


In [15]:
# import random

# import wandb

# run = wandb.init(
#     entity="21f2000522-indian-institute-of-technology-madras",
#     project="DeBERTa-v3-base",
#     config={
#         "learning_rate": 1e-5,
#         "architecture": "Transformer",
#         "dataset": "Kaggle dataset",
#         "epochs": 4,
#     },
# )

# Transformer training function

In [16]:
def run_transformer_training(
    model_name,
    save_path,
    train_dataframe,
    validation_dataframe,
    batch_size=2,
    epochs=4,
    learning_rate=1e-5,
    max_len=128
):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    model = AutoModelForMultipleChoice.from_pretrained(
    model_name,
    ignore_mismatched_sizes=True
)


    model = model.float().to(device)

    train_dataset = TransformerDataset(
        train_dataframe,
        tokenizer,
        max_len=max_len
    )

    validation_dataset = TransformerDataset(
        validation_dataframe,
        tokenizer,
        max_len=max_len
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=batch_size * 2,
        shuffle=False
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=0.01
    )

    total_steps = max(
        1,
        (len(train_loader) // GRAD_ACCUM_STEPS)
        * epochs
    )

    warmup_steps = max(
        1,
        total_steps // 10
    )

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        warmup_steps,
        total_steps
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=torch.cuda.is_available()
    )

    best_map3 = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            token_type_ids = batch.get("token_type_ids")

            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    labels=labels
                )

                loss = outputs.loss / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            if (
                (step + 1) % GRAD_ACCUM_STEPS == 0
                or (step + 1) == len(train_loader)
            ):
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

        model.eval()

        predictions = []
        targets = []

        with torch.no_grad():
            for batch in validation_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)

                token_type_ids = batch.get(
                    "token_type_ids"
                )

                if token_type_ids is not None:
                    token_type_ids = token_type_ids.to(device)

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )

                predictions.append(
                    outputs.logits.cpu().numpy()
                )

                targets.append(
                    labels.cpu().numpy()
                )

        predictions = np.concatenate(predictions)
        targets = np.concatenate(targets)

        map3_score = compute_map3(
            predictions,
            targets
        )

        accuracy = np.mean(
            np.argmax(
                predictions,
                axis=1
            ) == targets
        )

    #     run.log(
    #     {"Val_Accuracy": accuracy, "Val_MAP3":map3_score}
    # )

        print(
            f"Epoch {epoch}: "
            f"accuracy={accuracy:.4f}, "
            f"MAP@3={map3_score:.4f}"
        )

        if map3_score > best_map3:
            best_map3 = map3_score

            model.save_pretrained(
                save_path
            )

            tokenizer.save_pretrained(
                save_path
            )

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_map3



#  Run Model 2 and Model 3

Two pretrained transformer models are trained:

### Model 2: DeBERTa-v3-base

This is the required pretrained model.

### Model 3: RoBERTa-base

This is the additional model of choice.

Both models are trained across five folds. Their mean validation MAP@3 values are used for model comparison and final model selection.


In [18]:
# !pip install -q wandb

# from kaggle_secrets import UserSecretsClient
# import os, wandb

# api_key = UserSecretsClient().get_secret("Wandb_key")
# os.environ["Wandb_key"] = api_key

# wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

# Run DeBerta-v3-base  only

In [19]:
# import random

# import wandb

# run = wandb.init(
#     entity="21f2000522-indian-institute-of-technology-madras",
#     project="DeBERTa-v3-base Rerun",
#     config={
#         "learning_rate": 1e-5,
#         "architecture": "Transformer",
#         "dataset": "Kaggle dataset",
#         "epochs": 4,
#     },
# )

In [20]:
RUN_DEBERTA = True

deberta_fold_scores = []

if RUN_DEBERTA:

    print("\n" + "=" * 60)
    print("TRAINING DEBERTA-V3-BASE")
    print("=" * 60)

    for fold in range(5):

        print(
            f"\nDeBERTa Fold {fold + 1}/5"
        )

        train_fold = folds_data[fold]["train_aug"]
        validation_fold = folds_data[fold]["val_aug"]

        deberta_score = run_transformer_training(
            model_name=DEBERTA_MODEL_PATH,
            save_path=f"/kaggle/working/deberta_fold_{fold}",
            train_dataframe=train_fold,
            validation_dataframe=validation_fold,
            batch_size=TRANSFORMER_BATCH_SIZE,
            epochs=TRANSFORMER_EPOCHS,
            learning_rate=DEBERTA_LR,
            max_len=128
        )

        deberta_fold_scores.append(
            deberta_score
        )

        print(
            f"DeBERTa Fold {fold + 1} MAP@3: "
            f"{deberta_score:.4f}"
        )

        run.log(
        { "Debert_score":deberta_score}
    )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    best_deberta_map3 = float(
        np.mean(deberta_fold_scores)
    )

    print("\n" + "=" * 60)
    print(
        f"Mean DeBERTa MAP@3: "
        f"{best_deberta_map3:.4f}"
    )
    print("=" * 60)

else:
    best_deberta_map3 = 0.0

run.finish()


TRAINING DEBERTA-V3-BASE

DeBERTa Fold 1/5


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 3: accuracy=0.7383, MAP@3=0.8315


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 4: accuracy=0.6749, MAP@3=0.7952
DeBERTa Fold 5 MAP@3: 0.8315

Mean DeBERTa MAP@3: 0.8399


Debert_score,▇█▁▇▅
Debert_score,0.8315


In [ ]:
import random

import wandb

run = wandb.init(
    entity="21f2000522-indian-institute-of-technology-madras",
    project="Roberta-base",
    config={
        "learning_rate": 1e-5,
        "architecture": "Transformer",
        "dataset": "Kaggle dataset",
        "epochs": 4,
    },
)

# Run Roberta Only

In [ ]:
RUN_ROBERTA = True

roberta_fold_scores = []

if RUN_ROBERTA:

    print("\n" + "=" * 60)
    print("TRAINING ROBERTA-BASE")
    print("=" * 60)

    for fold in range(5):

        print(
            f"\nRoBERTa Fold {fold + 1}/5"
        )

        train_fold = folds_data[fold]["train_aug"]
        validation_fold = folds_data[fold]["val_aug"]

        roberta_score = run_transformer_training(
            model_name=ROBERTA_MODEL_PATH,
            save_path=f"/kaggle/working/roberta_fold_{fold}",
            train_dataframe=train_fold,
            validation_dataframe=validation_fold,
            batch_size=TRANSFORMER_BATCH_SIZE,
            epochs=TRANSFORMER_EPOCHS,
            learning_rate=ROBERTA_LR,
            max_len=128
        )

        roberta_fold_scores.append(
            roberta_score
        )

        print(
            f"RoBERTa Fold {fold + 1} MAP@3: "
            f"{roberta_score:.4f}"
        )

        # run.log({ "Debert_score":roberta_score})

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    best_roberta_map3 = float(
        np.mean(roberta_fold_scores)
    )

    print("\n" + "=" * 60)
    print(
        f"Mean RoBERTa MAP@3: "
        f"{best_roberta_map3:.4f}"
    )
    print("=" * 60)

else:
    best_roberta_map3 = 0.0

# run.finish()

In [ ]:
if RUN_TRANSFORMER_TRAINING:
    deberta_fold_scores = []
    roberta_fold_scores = []

    for fold in range(5):
        print("\n" + "=" * 60)
        print(f"TRANSFORMER FOLD {fold + 1}/5")
        print("=" * 60)

        train_fold = folds_data[fold]["train_aug"]
        validation_fold = folds_data[fold]["val_aug"]

        print("\nMODEL 2: DeBERTa-v3-base")

        deberta_score = run_transformer_training(
            model_name=DEBERTA_MODEL_PATH,
            save_path=f"/kaggle/working/deberta_fold_{fold}",
            train_dataframe=train_fold,
            validation_dataframe=validation_fold,
            batch_size=TRANSFORMER_BATCH_SIZE,
            epochs=TRANSFORMER_EPOCHS,
            learning_rate=DEBERTA_LR,
            max_len=128
        )

        deberta_fold_scores.append(
            deberta_score
        )

        print("\nMODEL 3: RoBERTa-base")

        roberta_score = run_transformer_training(
            model_name=ROBERTA_MODEL_PATH,
            save_path=f"/kaggle/working/roberta_fold_{fold}",
            train_dataframe=train_fold,
            validation_dataframe=validation_fold,
            batch_size=TRANSFORMER_BATCH_SIZE,
            epochs=TRANSFORMER_EPOCHS,
            learning_rate=ROBERTA_LR,
            max_len=128
        )

        roberta_fold_scores.append(
            roberta_score
        )

    best_deberta_map3 = float(
        np.mean(deberta_fold_scores)
    )

    best_roberta_map3 = float(
        np.mean(roberta_fold_scores)
    )

else:
    best_deberta_map3 = 0.0
    best_roberta_map3 = 0.0


run.finish()


# Compare all three models

In [ ]:

model_comparison = pd.DataFrame({
    "Model": [
        "BiGRU",
        "DeBERTa-v3-base",
        "RoBERTa-base"
    ],
    "Model Type": [
        "Built from scratch",
        "Pretrained model",
        "Additional model"
    ],
    "Validation MAP@3": [
        best_bigru_map3,
        best_deberta_map3,
        best_roberta_map3
    ]
})

model_comparison = model_comparison.sort_values(
    "Validation MAP@3",
    ascending=False
).reset_index(drop=True)

display(model_comparison)


#  Select only the strongest pretrained model(s)


In [ ]:
transformer_scores = {
    "deberta": best_deberta_map3,
    "roberta": best_roberta_map3
}

usable_transformers = {
    model_name: score
    for model_name, score in transformer_scores.items()
    if score >= MIN_USABLE_MAP3
}

if len(usable_transformers) == 0:
    best_name = max(
        transformer_scores,
        key=transformer_scores.get
    )

    selected_models = {
        best_name: transformer_scores[best_name]
    }

elif len(usable_transformers) == 1:
    selected_models = usable_transformers

else:
    selected_models = dict(
        sorted(
            usable_transformers.items(),
            key=lambda item: item[1],
            reverse=True
        )[:2]
    )

total_score = sum(
    selected_models.values()
)

ensemble_weights = {
    model_name: score / total_score
    for model_name, score in selected_models.items()
}

print("Selected final models:")
for model_name, weight in ensemble_weights.items():
    print(f"{model_name}: {weight:.4f}")


# Generate transformer probabilities

In [ ]:
def get_transformer_predictions(
    model_folder,
    dataframe,
    max_len=128
):
    tokenizer = AutoTokenizer.from_pretrained(
        model_folder
    )

    model = AutoModelForMultipleChoice.from_pretrained(
        model_folder
    ).to(device)

    model.eval()

    dataset = TransformerDataset(
        dataframe,
        tokenizer,
        max_len=max_len,
        is_test=True
    )

    loader = DataLoader(
        dataset,
        batch_size=TRANSFORMER_BATCH_SIZE * 2,
        shuffle=False
    )

    probabilities = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            token_type_ids = batch.get(
                "token_type_ids"
            )

            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )

            probabilities.append(
                torch.softmax(
                    outputs.logits,
                    dim=1
                ).cpu().numpy()
            )

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.concatenate(
        probabilities
    )


# Combine selected model predictions

In [ ]:
deberta_probabilities = []
roberta_probabilities = []

for fold in range(5):
    print(f"Generating test predictions for fold {fold + 1}...")

    if "deberta" in ensemble_weights:
        deberta_probabilities.append(
            get_transformer_predictions(
                f"/kaggle/working/deberta_fold_{fold}",
                test_df_aug
            )
        )

    if "roberta" in ensemble_weights:
        roberta_probabilities.append(
            get_transformer_predictions(
                f"/kaggle/working/roberta_fold_{fold}",
                test_df_aug
            )
        )

model_probabilities = {}

if len(deberta_probabilities) > 0:
    model_probabilities["deberta"] = np.mean(
        deberta_probabilities,
        axis=0
    )

if len(roberta_probabilities) > 0:
    model_probabilities["roberta"] = np.mean(
        roberta_probabilities,
        axis=0
    )

probs_ensemble = np.zeros(
    (len(test_df), 5),
    dtype=np.float32
)

for model_name, weight in ensemble_weights.items():
    probs_ensemble += (
        weight
        * model_probabilities[model_name]
    )

probs_ensemble = probs_ensemble / np.clip(
    probs_ensemble.sum(
        axis=1,
        keepdims=True
    ),
    1e-10,
    None
)

print("Final probability shape:", probs_ensemble.shape)


# Apply exact-prompt correction

In [ ]:
def clean_prompt_lookup(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


train_lookup = {}

for _, row in train_df.iterrows():
    train_lookup[
        clean_prompt_lookup(row["prompt"])
    ] = row


replaced_count = 0

for test_index, test_row in test_df.iterrows():
    cleaned_prompt = clean_prompt_lookup(
        test_row["prompt"]
    )

    if cleaned_prompt in train_lookup:
        train_row = train_lookup[
            cleaned_prompt
        ]

        correct_answer_column = train_row[
            "answer"
        ]

        correct_answer_text = str(
            train_row[
                correct_answer_column
            ]
        )

        test_options = [
            str(test_row[column])
            for column in ["A", "B", "C", "D", "E"]
        ]

        vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5)
        )

        vectors = vectorizer.fit_transform(
            test_options
            + [correct_answer_text]
        )

        similarities = cosine_similarity(
            vectors[-1],
            vectors[:-1]
        )[0]

        best_option_index = int(
            np.argmax(similarities)
        )

        probs_ensemble[test_index] = 0.0
        probs_ensemble[
            test_index,
            best_option_index
        ] = 1.0

        replaced_count += 1


print(
    "Exact prompt predictions replaced:",
    replaced_count
)

# Review validation performance before submission

In [ ]:
# Print BiGRU validation results

print("\n" + "=" * 50)
print("BiGRU Validation Results")
print("=" * 50)

print(f"Validation Accuracy : {best_bigru_accuracy:.4f}")
print(f"Validation MAP@3    : {best_bigru_map3:.4f}")




In [ ]:
print("\n" + "=" * 50)
print("DeBERTa Fold MAP@3 Scores")
print("=" * 50)

for fold_number, fold_score in enumerate(
    deberta_fold_scores,
    start=1
):
    print(
        f"Fold {fold_number}: "
        f"{fold_score:.4f}"
    )


print("\n" + "=" * 50)
print("RoBERTa Fold MAP@3 Scores")
print("=" * 50)

for fold_number, fold_score in enumerate(
    roberta_fold_scores,
    start=1
):
    print(
        f"Fold {fold_number}: "
        f"{fold_score:.4f}"
    )


In [ ]:
# Calculate and print mean MAP@3 for each model

mean_deberta_map3 = float(
    np.mean(deberta_fold_scores)
)

mean_roberta_map3 = float(
    np.mean(roberta_fold_scores)
)

overall_mean_map3 = float(
    np.mean([
        best_bigru_map3,
        mean_deberta_map3,
        mean_roberta_map3
    ])
)


print("\n" + "=" * 50)
print("MEAN AVERAGE PRECISION AT 3")
print("=" * 50)

print(
    f"BiGRU MAP@3   : "
    f"{best_bigru_map3:.4f}"
)

print(
    f"DeBERTa MAP@3 : "
    f"{mean_deberta_map3:.4f}"
)

print(
    f"RoBERTa MAP@3 : "
    f"{mean_roberta_map3:.4f}"
)

print(
    f"Overall mean MAP@3 across three models: "
    f"{overall_mean_map3:.4f}"
)

#  Compare all three models

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "BiGRU",
        "DeBERTa-v3-base",
        "RoBERTa-base"
    ],

    "Model Type": [
        "Built from scratch",
        "Pretrained model",
        "Additional model"
    ],

    "Mean Validation MAP@3": [
        best_bigru_map3,
        mean_deberta_map3,
        mean_roberta_map3
    ],

    "Used for Final Prediction": [
        "No",
        "Yes" if "deberta" in ensemble_weights else "No",
        "Yes" if "roberta" in ensemble_weights else "No"
    ]
})

model_comparison = model_comparison.sort_values(
    by="Mean Validation MAP@3",
    ascending=False
).reset_index(drop=True)

display(model_comparison)


# Pre Submission Validation

In [ ]:
print("\n" + "=" * 50)
print("PRE-SUBMISSION CHECK")
print("=" * 50)

print("Selected model weights:")

for model_name, model_weight in ensemble_weights.items():
    print(
        f"{model_name}: "
        f"{model_weight:.4f}"
    )

print(
    "\nFinal probability array shape:",
    probs_ensemble.shape
)

print(
    "Expected probability shape:",
    (len(test_df), 5)
)

print(
    "All probability values are finite:",
    np.isfinite(probs_ensemble).all()
)

print(
    "Average row probability sum:",
    probs_ensemble.sum(axis=1).mean()
)

assert probs_ensemble.shape == (len(test_df), 5)
assert np.isfinite(probs_ensemble).all()

print("\nAll checks passed. Ready to create submission.csv")


Creating the final submission

In [ ]:
top3_indices = np.argsort(
    probs_ensemble,
    axis=1
)[:, ::-1][:, :3]

predictions = [
    " ".join(
        OPTIONS[row]
    )
    for row in top3_indices
]

id_column = (
    "ID"
    if "ID" in test_df.columns
    else "id"
)

submission = pd.DataFrame({
    "ID": test_df[id_column],
    "Prediction": predictions
})

submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("Submission saved:", submission_path)
print("Submission shape:", submission.shape)

display(submission.head(10))

Milestone-1

In [ ]:
# import pandas as pd
# import string
# import numpy as np

# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity


# x1 = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# x2 = x1["answer"].value_counts()
# print("Q1")
# print(x2)
# x3 = x2.max() + x2.min()
# print(x3)
# z1 = set()

# for a1 in x1["prompt"]:

#     a1 = str(a1).lower()
#     for b1 in string.punctuation:
#         a1 = a1.replace(b1, "")
#     a1 = a1.split()
#     for c1 in a1:
#         z1.add(c1)

# print("Q2")
# print(len(z1))


# r1 = x1.iloc[0]["prompt"]
# r1 = str(r1).lower()
# for d1 in string.punctuation:
#     r1 = r1.replace(d1, "")
# r1 = r1.split()
# r2 = []
# for e1 in r1:
#     if e1 not in ENGLISH_STOP_WORDS:
#         r2.append(e1)
# print("Q3")
# print(len(r2))


# k2 = (
#     x1["prompt"].fillna("") + " " +
#     x1["A"].fillna("") + " " +
#     x1["B"].fillna("") + " " +
#     x1["C"].fillna("") + " " +
#     x1["D"].fillna("") + " " +
#     x1["E"].fillna("")
# )
# m1 = TfidfVectorizer(stop_words="english")
# m2 = m1.fit_transform(k2)
# print("Q4")
# print(len(m1.get_feature_names_out()))


# p1 = str(x1.iloc[0]["prompt"])
# p2 = str(x1.iloc[0]["A"])
# p3 = m1.transform([p1, p2])
# p4 = cosine_similarity(p3[0], p3[1])[0][0]
# print("Q5")
# print(round(p4, 4))


# c1 = 0
# for _, y1 in x1.iterrows():
#     s1 = {}
#     q1 = m1.transform([str(y1["prompt"])])
#     for q2 in ["A", "B", "C", "D", "E"]:
#         q3 = m1.transform([str(y1[q2])])
#         q4 = cosine_similarity(q1, q3)[0][0]
#         s1[q2] = q4
#     q5 = max(s1, key=s1.get)

#     if q5 == y1["answer"]:
#         c1 = c1 + 1
# print("Q6")
# print((c1 / len(x1)) * 100)


# def abc(a, b):
#     if a in b:
#         return 1 / (b.index(a) + 1)
#     return 0
# print("Q7")
# print(abc("C", ["C", "A", "B"]))


# print("Q8")
# print(abc("B", ["D", "B", "E"]))


# t1 = x1["answer"].value_counts()
# t2 = list(t1.index[:3])
# v1 = []
# for g1 in x1["answer"]:
#     v1.append(abc(g1, t2))
# print("Q9")
# print(np.mean(v1))


# v2 = []
# for _, h1 in x1.iterrows():
#     h2 = {}
#     h3 = m1.transform([str(h1["prompt"])])
#     for h4 in ["A", "B", "C", "D", "E"]:

#         h5 = m1.transform([str(h1[h4])])
#         h6 = cosine_similarity(h3, h5)[0][0]
#         h2[h4] = h6

#     h7 = sorted(h2.items(), key=lambda x: x[1], reverse=True)
#     h8 = []

#     for h9 in h7[:3]:
#         h8.append(h9[0])

#     v2.append(abc(h1["answer"], h8))

# print("Q10")
# print(np.mean(v2))